## Vis for labels: OrigImg, GT, Preds
### This part is for YYF_30Case data and WSSS_Unet model

In [1]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
from os.path import join

In [2]:
data_root_dir = "../data/YYF_30Case"
pred_root_dir = "../experiments/wsss_unet/results/YYF_30Case"

save_dir = join(pred_root_dir, "imgs_vis")
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
# preprocessed images
img_dir = join(data_root_dir, "preprocessed_size256")
img_cropped_dir = join(data_root_dir, "preprocessed_size256_cropped")
# our prediction
pred_dir = join(pred_root_dir, "preprocessed_size256/pred_mask")
pred_cropped_dir = join(pred_root_dir, "preprocessed_size256_cropped/pred_mask")
# labels from sheng zhang and YYF
labels_1_dir = join(data_root_dir, "labels_1_imgs")
labels_2_dir = join(data_root_dir, "labels_2_imgs")

all_dirs = [
    img_dir,
    img_cropped_dir,
    pred_dir,
    pred_cropped_dir,
    labels_1_dir,
    labels_2_dir
]
# check if all directories exist
for d in all_dirs:
    if not os.path.exists(d):
        print(f"Directory {d} does not exist.")
        exit(1)

In [3]:

def get_img_list(img_dir: str, case_index: int = 0) -> dict[list[str]]:
    """
    Get the list of images in the specified directory.
    Args:
        img_dir (str): Directory containing the images.
        case_index (int): Index of the case to process.
    Returns:
        dict: Dictionary containing the image names and paths.
    """
    # Get the list of cases
    cases_list = os.listdir(img_dir)
    cases_list = [f for f in cases_list if '.' not in f]
    cases_list.sort()
    
    case_id = cases_list[case_index]

    imgs = {}
    imgs['name'] = sorted(os.listdir(join(img_dir, case_id)))
    imgs['path'] = [join(img_dir, case_id, f) for f in imgs['name']]
    imgs['case_id'] = case_id
    # print("img name", imgs['name'][:3])
    # print("img path", imgs['path'][:3])
    
    return imgs
    
def load_img(img_path_list: list[str]) -> np.ndarray:
    """
    Load images from the specified paths.
    Args:
        img_path_list (list[str]): List of image paths.
    Returns:
        np.ndarray: Array of loaded images.
    """
    img_list = []
    for img_path in img_path_list:
        img = Image.open(img_path)
        img = np.array(img)
        img_list.append(img)
    img = np.stack(img_list, axis=0)
    return img

def vis_imgs(save_dir, img_list: list[np.ndarray], middle_slice_id: int = 0, case_id: str = "case_0", show_cropped: bool = False, show: bool = False) -> None:
    """
    Visualize images in a grid.

    Args:
        save_dir (str): Directory to save the visualizations.
        img_list (list[np.ndarray]): List of image arrays to visualize.
        middle_slice_id (int): Starting slice index for visualization.
        case_id (str): Identifier for the case being visualized.
        show_cropped (bool): Whether to include cropped images in the visualization.
        show (bool): Whether to display the plot interactively.
    """
    all_titles = ["Original", "Cropped", "Pred", "Pred_cropped", "Labels_1", "Labels_2"]
    if show_cropped:
        all_imgs = img_list[1:2] + img_list[3:4]
        all_titles = [strs for strs in all_titles if "cropped" in strs.lower()]
    else:
        all_imgs = img_list[:1] + img_list[2:3] + img_list[4:]  # Exclude cropped images
        all_titles = [strs for strs in all_titles if "cropped" not in strs.lower()]

    fig, axes = plt.subplots(len(all_imgs), 8, figsize=(20, len(all_imgs) * 3))
    for i in range(8):
        for j, imgs in enumerate(all_imgs):
            axes[j, i].imshow(imgs[i], cmap='gray')
            axes[j, i].set_title(all_titles[j] + f" {middle_slice_id + i}")
            axes[j, i].axis('off')

    plt.suptitle(f"Case {case_id} - Slices {middle_slice_id} to {middle_slice_id + 8}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.99])  # Adjust layout to fit the suptitle

    img_name = f"case_{case_id}_slice_{middle_slice_id}_cropped.png" if show_cropped else f"case_{case_id}_slice_{middle_slice_id}.png"
    plt.savefig(join(save_dir, img_name), dpi=300)
    print(f"Saved figure for case {case_id} at slice {middle_slice_id}.")

    plt.show() if show else plt.close()


In [4]:
# img_list = get_img_list(img_dir)
# img_cropped_list = get_img_list(img_cropped_dir)

all_img_list = [get_img_list(img_dir, case_index = 1) for img_dir in all_dirs]
img_list, img_cropped_list, pred_list, pred_cropped_list, labels_1_list, labels_2_list = all_img_list
case_id = img_list['case_id']

middle_slice_idx = len(img_list['name']) // 2
for middle_slice_id in range(middle_slice_idx - 100, middle_slice_idx + 101, 50):
    slice_name = img_list['name'][middle_slice_id]
    print(f"processing case_id: {case_id}; slices: {middle_slice_id}; slice_name: {slice_name}")

    # get the middle slice name of the cropped image, as the cropped image doest have top and bottom k imgs which does not have lung 
    try:
        converted_id = img_cropped_list['name'].index(slice_name)
    except:
        converted_id = img_cropped_list['name'].index(slice_name.replace("_0000", ""))


    
    imgs_np, pred_np, labels_1_np, labels_2_np = [load_img(list_name['path'][middle_slice_id: middle_slice_id+8]) for list_name in [img_list, pred_list, labels_1_list, labels_2_list]]    
    imgs_cropped_np, pred_cropped_np = [load_img(list_name['path'][converted_id: converted_id+8]) for list_name in [img_cropped_list, pred_cropped_list]]

    all_imgs = [imgs_np, imgs_cropped_np, pred_np, pred_cropped_np, labels_1_np, labels_2_np]
    vis_imgs(save_dir, all_imgs, middle_slice_id, case_id, show_cropped=False)
    vis_imgs(save_dir, all_imgs, middle_slice_id, case_id, show_cropped=True)
    # break


processing case_id: 2_0000; slices: 106; slice_name: 2_0000_106.png
Saved figure for case 2_0000 at slice 106.
Saved figure for case 2_0000 at slice 106.
processing case_id: 2_0000; slices: 156; slice_name: 2_0000_156.png
Saved figure for case 2_0000 at slice 156.
Saved figure for case 2_0000 at slice 156.
processing case_id: 2_0000; slices: 206; slice_name: 2_0000_206.png
Saved figure for case 2_0000 at slice 206.
Saved figure for case 2_0000 at slice 206.
processing case_id: 2_0000; slices: 256; slice_name: 2_0000_256.png
Saved figure for case 2_0000 at slice 256.
Saved figure for case 2_0000 at slice 256.
processing case_id: 2_0000; slices: 306; slice_name: 2_0000_306.png
Saved figure for case 2_0000 at slice 306.
Saved figure for case 2_0000 at slice 306.
